In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load and describe the data
* Load the data (in wide format) and convert to long format
* Merge the 2 time series and filter to the time window of interest (1992 - 2024)
* Drop regional/income-groups with now regions

In [2]:
# ── 1. Load ────────────────────────────────────────────────────────────────────
co2_wide = pd.read_csv("data/API_EN.GHG.CO2.PC.CE.AR5_DS2_en_csv_v2_610.csv", skiprows=4)
gdp_wide = pd.read_csv("data/API_NY.GDP.PCAP.PP.CD_DS2_en_csv_v2_35.csv", skiprows=4)
meta     = pd.read_csv("data/Metadata_Country_API_EN.GHG.CO2.PC.CE.AR5_DS2_en_csv_v2_610.csv")

# ── 2. Wide → Long ─────────────────────────────────────────────────────────────
id_vars = ["Country Name", "Country Code", "Indicator Name", "Indicator Code"]
year_cols = [c for c in co2_wide.columns if c.isdigit() and int(c) <= 2024]

co2 = (co2_wide
       .melt(id_vars=id_vars, value_vars=year_cols, var_name="year", value_name="co2_pc")
       .assign(year=lambda x: x["year"].astype(int)))

gdp = (gdp_wide
       .melt(id_vars=id_vars, value_vars=year_cols, var_name="year", value_name="gdp_pc")
       .assign(year=lambda x: x["year"].astype(int)))

# ── 3. Merge + filter to assignment window (1992–latest) ──────────────────────
df = (co2[["Country Code", "Country Name", "year", "co2_pc"]]
      .merge(gdp[["Country Code", "year", "gdp_pc"]], on=["Country Code", "year"])
      .merge(meta[["Country Code", "Region", "IncomeGroup"]], on="Country Code")
      .query("year >= 1992")
      .rename(columns={"Country Name": "country", "Country Code": "iso3"}))

# drop aggregates (World Bank includes regional/income-group rows with no Region)
df = df[df["Region"].notna()].copy()

In [3]:
# ── 4. Summary statistics ──────────────────────────────────────────────────────
print(f"Countries: {df['iso3'].nunique()}")
print(f"Years:     {df['year'].min()} – {df['year'].max()}")
print(f"Obs:       {len(df):,}")
print(f"\nMissing co2_pc: {df['co2_pc'].isna().mean():.2%}")
print(f"Missing gdp_pc: {df['gdp_pc'].isna().mean():.2%}")
print("\nSummary statistics:")
print(df[["co2_pc", "gdp_pc"]].describe().round(2))

Countries: 217
Years:     1992 – 2024
Obs:       7,161

Missing co2_pc: 6.45%
Missing gdp_pc: 9.80%

Summary statistics:
        co2_pc     gdp_pc
count  6699.00    6459.00
mean      4.85   17912.89
std       9.40   21596.41
min       0.00     254.39
25%       0.52    3259.44
50%       2.19    9469.67
75%       6.13   24781.25
max     202.87  180939.44


# Dropping countries with low data coverage

In [4]:
total_years = df["year"].nunique()

coverage = (df.groupby(["iso3", "country"])[["co2_pc", "gdp_pc"]]
              .apply(lambda x: x.notna().all(axis=1).sum())
              .rename("both_obs")
              .reset_index()
              .assign(coverage_pct=lambda x: x["both_obs"] / total_years * 100))

print(coverage["coverage_pct"].describe().round(1))
print(f"\nCountries below 80% coverage: {(coverage['coverage_pct'] < 80).sum()}")
print(f"Countries below 50% coverage: {(coverage['coverage_pct'] < 50).sum()}")

count    217.0
mean      87.2
std       31.8
min        0.0
25%      100.0
50%      100.0
75%      100.0
max      100.0
Name: coverage_pct, dtype: float64

Countries below 80% coverage: 33
Countries below 50% coverage: 25


In [5]:
# ── Drop countries with below 80% coverage ────────────────────────────────────────────────
THRESHOLD = 50

good_coverage = coverage.loc[coverage["coverage_pct"] >= THRESHOLD, "iso3"]
df_clean = df[df["iso3"].isin(good_coverage)].copy()

dropped = coverage[coverage["coverage_pct"] < THRESHOLD][["iso3", "country", "coverage_pct"]].sort_values("coverage_pct")
print(f"\nDropped {len(dropped)} countries:")
print(dropped.to_string(index=False))
print(f"\nRemaining: {df_clean['iso3'].nunique()} countries, {len(df_clean):,} obs")


Dropped 25 countries:
iso3                   country  coverage_pct
 AND                   Andorra      0.000000
 ASM            American Samoa      0.000000
 CHI           Channel Islands      0.000000
 CUB                      Cuba      0.000000
 CUW                   Curacao      0.000000
 GIB                 Gibraltar      0.000000
 GUM                      Guam      0.000000
 IMN               Isle of Man      0.000000
 MNE                Montenegro      0.000000
 LIE             Liechtenstein      0.000000
 MAF  St. Martin (French part)      0.000000
 MCO                    Monaco      0.000000
 NCL             New Caledonia      0.000000
 MNP  Northern Mariana Islands      0.000000
 PRK Korea, Dem. People's Rep.      0.000000
 PSE        West Bank and Gaza      0.000000
 SSD               South Sudan      0.000000
 PYF          French Polynesia      0.000000
 SMR                San Marino      0.000000
 SRB                    Serbia      0.000000
 VGB    British Virgin Islands  

# Estimate and interpret coefficients for each model

## 1. Cross-section OLS for year 2005

In [6]:
# create a helper function for cross section OLS of a year
import statsmodels.formula.api as smf

def cross_section_ols(df, year):
    d = (df[df["year"] == year][["iso3", "country", "co2_pc", "gdp_pc"]]
           .dropna()
           .query("co2_pc > 0 and gdp_pc > 0")    # only include countries with both positive GDP and CO2 per capita
           .assign(log_co2=lambda x: np.log(x["co2_pc"]),
                   log_gdp=lambda x: np.log(x["gdp_pc"]))
        )

    model = smf.ols("log_co2 ~ log_gdp", data=d).fit(cov_type="HC3")

    # manually build results table instead of calling summary()
    results_table = pd.DataFrame({
        "coef":    model.params,
        "HC3_se":  model.bse,
        "t":       model.tvalues,
        "p-value": model.pvalues,
        "CI_low":  model.conf_int()[0],
        "CI_high": model.conf_int()[1],
    }).round(4)

    print(f"\n── Cross-section OLS: {year} ── N={len(d)} countries ──")
    print(f"R²: {model.rsquared:.3f}")
    print(results_table)

    return model

In [7]:
# run the model for 2005
model_2005 = cross_section_ols(df_clean, 2005)


── Cross-section OLS: 2005 ── N=187 countries ──
R²: 0.640
              coef  HC3_se        t  p-value   CI_low  CI_high
Intercept -10.4256  0.5964 -17.4806      0.0 -11.5946  -9.2567
log_gdp     1.2180  0.0679  17.9320      0.0   1.0849   1.3512


## 2. Cross-section OLS for latest available year

In [8]:
# run the model for 2024
model_2024 = cross_section_ols(df_clean, 2024)


── Cross-section OLS: 2024 ── N=179 countries ──
R²: 0.634
             coef  HC3_se        t  p-value   CI_low  CI_high
Intercept -9.7603  0.6405 -15.2393      0.0 -11.0156  -8.5050
log_gdp    1.0599  0.0646  16.4062      0.0   0.9333   1.1865


## 3. First difference model, with time trend, no lags

In [9]:
from linearmodels.panel import PanelOLS

def first_diff_model(df, lags=0):
    """
    First difference model using PanelOLS.
    TimeEffects replaces C(year) dummies.
    """
    # ── prep ──────────────────────────────────────────────────────────────────
    d = (df.query("co2_pc > 0 and gdp_pc > 0")
           .assign(log_co2=lambda x: np.log(x["co2_pc"]),
                   log_gdp=lambda x: np.log(x["gdp_pc"]))
           .sort_values(["iso3", "year"]))

    # ── first differences within each country ─────────────────────────────────
    d["dlog_co2"] = d.groupby("iso3")["log_co2"].diff()
    d["dlog_gdp"] = d.groupby("iso3")["log_gdp"].diff()

    # ── lags ──────────────────────────────────────────────────────────────────
    formula_vars = "dlog_gdp"
    if lags > 0:
        for lag in range(1, lags + 1):
            d[f"dlog_gdp_lag{lag}"] = d.groupby("iso3")["dlog_gdp"].shift(lag)
            formula_vars += f" + dlog_gdp_lag{lag}"

    # ── dropna + set MultiIndex (required by linearmodels) ────────────────────
    drop_cols = ["dlog_co2", "dlog_gdp"] + [f"dlog_gdp_lag{lag}" for lag in range(1, lags+1)]
    d = (d.dropna(subset=drop_cols)
          .set_index(["iso3", "year"]))  # PanelOLS requires MultiIndex

    model = PanelOLS.from_formula(
        f"dlog_co2 ~ {formula_vars} + TimeEffects",
        data=d
    ).fit(cov_type="clustered", cluster_entity=True)

    print(f"\n── FD Model (PanelOLS) | lags={lags} ──")
    print(f"N obs: {model.nobs:,}  |  N countries: {d.index.get_level_values('iso3').nunique()}")
    print(f"R² (within): {model.rsquared:.3f}")
    print(model.summary.tables[1])

    return model

In [10]:
model_fd0 = first_diff_model(df_clean, lags=0)


── FD Model (PanelOLS) | lags=0 ──
N obs: 5,971  |  N countries: 190
R² (within): 0.023
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
dlog_gdp       0.3195     0.0481     6.6417     0.0000      0.2252      0.4138


## 4. First difference model, with time trend, 2 year lags

In [11]:
model_fd2 = first_diff_model(df_clean, lags=2)


── FD Model (PanelOLS) | lags=2 ──
N obs: 5,591  |  N countries: 190
R² (within): 0.019
                               Parameter Estimates                               
               Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
---------------------------------------------------------------------------------
dlog_gdp          0.2708     0.0482     5.6138     0.0000      0.1762      0.3653
dlog_gdp_lag1     0.0191     0.0472     0.4057     0.6850     -0.0734      0.1116
dlog_gdp_lag2     0.0516     0.0214     2.4120     0.0159      0.0097      0.0935


## 5. First difference model, with time trend, 6 year lags

In [12]:
model_fd6 = first_diff_model(df_clean, lags=6)


── FD Model (PanelOLS) | lags=6 ──
N obs: 4,831  |  N countries: 190
R² (within): 0.019
                               Parameter Estimates                               
               Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
---------------------------------------------------------------------------------
dlog_gdp          0.2687     0.0520     5.1675     0.0000      0.1667      0.3706
dlog_gdp_lag1     0.0381     0.0495     0.7709     0.4408     -0.0588      0.1351
dlog_gdp_lag2     0.0202     0.0246     0.8199     0.4123     -0.0281      0.0685
dlog_gdp_lag3     0.0232     0.0344     0.6737     0.5005     -0.0443      0.0906
dlog_gdp_lag4     0.0410     0.0342     1.2006     0.2300     -0.0260      0.1081
dlog_gdp_lag5    -0.0366     0.0370    -0.9896     0.3224     -0.1091      0.0359
dlog_gdp_lag6     0.0457     0.0322     1.4172     0.1565     -0.0175      0.1088


## 6. Fixed effects model with time and country fixed effects

In [13]:
# from linearmodels.panel import PanelOLS

def fixed_effects_model(df):
    """
    Fixed effects model with time and country fixed effects.
    Equivalent to: log_co2 ~ log_gdp + EntityEffects + TimeEffects
    """
    # ── prep ──────────────────────────────────────────────────────────────────
    d = (df.query("co2_pc > 0 and gdp_pc > 0")
           .assign(log_co2=lambda x: np.log(x["co2_pc"]),
                   log_gdp=lambda x: np.log(x["gdp_pc"]))
           .dropna(subset=["log_co2", "log_gdp"])
           .set_index(["iso3", "year"]))  # linearmodels requires MultiIndex

    model = PanelOLS.from_formula(
        "log_co2 ~ log_gdp + EntityEffects + TimeEffects",
        data=d
    ).fit(cov_type="clustered", cluster_entity=True)

    print("\n── Fixed Effects Model | Country + Time FE ──")
    print(f"N obs: {model.nobs:,}  |  N countries: {d.index.get_level_values('iso3').nunique()}")
    print(f"R² (within): {model.rsquared:.3f}")
    print(model.summary.tables[1])

    return model

In [14]:
model_fe = fixed_effects_model(df_clean)


── Fixed Effects Model | Country + Time FE ──
N obs: 6,161  |  N countries: 190
R² (within): 0.069
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
log_gdp        0.4414     0.0747     5.9105     0.0000      0.2950      0.5878


# Potential confounder: Industry share of GDP (%)
A country's industrial activities determine how CO2-intensive its economic activity is. In particular, an economy growing more through industrial activities like steel, cement, or chemicals might emit far more per dollar of output than one growing through finance, services, or tourism. 

Industry share is also correlated with GDP because early-stage development is typically driven by manufacturing, while richer countries have largely transitioned to services.

## Prepare the dataset with the confounder variable included (`industry_share`)

In [15]:
# ── Load raw data ──────────────────────────────────────────────────────────────
co2_wide      = pd.read_csv("data/API_EN.GHG.CO2.PC.CE.AR5_DS2_en_csv_v2_610.csv", skiprows=4)
gdp_wide      = pd.read_csv("data/API_NY.GDP.PCAP.PP.CD_DS2_en_csv_v2_35.csv", skiprows=4)
industry_wide = pd.read_csv("data/API_NV.IND.TOTL.ZS_DS2_en_csv_v2_36.csv", skiprows=4)
meta          = pd.read_csv("data/Metadata_Country_API_EN.GHG.CO2.PC.CE.AR5_DS2_en_csv_v2_610.csv")

# ── Wide → Long ────────────────────────────────────────────────────────────────
id_vars   = ["Country Name", "Country Code", "Indicator Name", "Indicator Code"]
year_cols = [c for c in co2_wide.columns if c.isdigit() and int(c) <= 2024]

def wide_to_long(wide_df, value_name):
    return (wide_df
            .melt(id_vars=id_vars, value_vars=year_cols, var_name="year", value_name=value_name)
            .assign(year=lambda x: x["year"].astype(int))
            [["Country Code", "year", value_name]])

co2      = wide_to_long(co2_wide,      "co2_pc")
gdp      = wide_to_long(gdp_wide,      "gdp_pc")
industry = wide_to_long(industry_wide, "industry_share")

# ── Merge ──────────────────────────────────────────────────────────────────────
df = (co2
      .merge(gdp,      on=["Country Code", "year"])
      .merge(industry, on=["Country Code", "year"])
      .merge(meta[["Country Code", "Region", "IncomeGroup"]], on="Country Code")
      .merge(co2_wide[["Country Code", "Country Name"]].drop_duplicates(), on="Country Code")
      .rename(columns={"Country Name": "country", "Country Code": "iso3"})
      .query("year >= 1992")
)

# drop aggregates
df = df[df["Region"].notna()].copy()

print(f"Countries: {df['iso3'].nunique()}")
print(f"Years:     {df['year'].min()} – {df['year'].max()}")
print(f"Obs:       {len(df):,}")
print(f"\nMissing co2_pc:        {df['co2_pc'].isna().sum():,}")
print(f"Missing gdp_pc:        {df['gdp_pc'].isna().sum():,}")
print(f"Missing industry_share:{df['industry_share'].isna().sum():,}")

# ── Coverage filter ────────────────────────────────────────────────────────────
total_years = df["year"].nunique()

coverage = (df.groupby(["iso3", "country"])[["co2_pc", "gdp_pc", "industry_share"]]
              .apply(lambda x: x.notna().all(axis=1).sum())
              .rename("both_obs")
              .reset_index()
              .assign(coverage_pct=lambda x: x["both_obs"] / total_years * 100))

THRESHOLD = 50
good_coverage = coverage.loc[coverage["coverage_pct"] >= THRESHOLD, "iso3"]
df_clean_ind = df[df["iso3"].isin(good_coverage)].copy()

dropped = (coverage[coverage["coverage_pct"] < THRESHOLD]
           [["iso3", "country", "coverage_pct"]]
           .sort_values("coverage_pct"))

print(f"\nDropped {len(dropped)} countries below {THRESHOLD}% coverage")
print(f"\nDropped {len(dropped)} countries:")
print(dropped.to_string(index=False))
print(f"Remaining: {df_clean['iso3'].nunique()} countries, {len(df_clean):,} obs")

Countries: 217
Years:     1992 – 2024
Obs:       7,161

Missing co2_pc:        462
Missing gdp_pc:        702
Missing industry_share:1,038

Dropped 32 countries below 50% coverage

Dropped 32 countries:
iso3                   country  coverage_pct
 AND                   Andorra      0.000000
 ASM            American Samoa      0.000000
 CUB                      Cuba      0.000000
 CHI           Channel Islands      0.000000
 CUW                   Curacao      0.000000
 IMN               Isle of Man      0.000000
 GUM                      Guam      0.000000
 GIB                 Gibraltar      0.000000
 MCO                    Monaco      0.000000
 MAF  St. Martin (French part)      0.000000
 MNE                Montenegro      0.000000
 LIE             Liechtenstein      0.000000
 PRK Korea, Dem. People's Rep.      0.000000
 NRU                     Nauru      0.000000
 NCL             New Caledonia      0.000000
 MNP  Northern Mariana Islands      0.000000
 VGB    British Virgin Islands  

In [16]:
df_clean_ind.head()

,iso3,year,co2_pc,gdp_pc,industry_share,Region,IncomeGroup,country
8480,ABW,1992,3.634519,23889.045018,NaN,Latin America & Caribbean,High income,Aruba
8482,AFG,1992,0.136358,NaN,NaN,Middle East & North Africa,Low income,Afghanistan
8484,AGO,1992,0.975473,3486.206867,NaN,Sub-Saharan Africa,Lower middle income,Angola
8485,ALB,1992,0.727802,1899.258869,NaN,Europe & Central Asia,Upper middle income,Albania
8488,ARE,1992,29.565193,87509.043492,55.298023,Middle East & North Africa,High income,United Arab Emirates


## 1. Cross-sectional OLS for 2005

In [17]:
# create a helper function for cross section OLS of a year
import statsmodels.formula.api as smf

def cross_section_ols(df, year):
    d = (df[df["year"] == year][["iso3", "country", "co2_pc", "gdp_pc", "industry_share"]]
           .dropna()
           .query("co2_pc > 0 and gdp_pc > 0 and industry_share > 0")    # only include countries with both positive GDP and CO2 per capita
           .assign(log_co2=lambda x: np.log(x["co2_pc"]),
                   log_gdp=lambda x: np.log(x["gdp_pc"]),
                   log_industry_share=lambda x: np.log(x['industry_share']),
                   )
        )

    model = smf.ols("log_co2 ~ log_gdp + log_industry_share", data=d).fit(cov_type="HC3")

    # manually build results table instead of calling summary()
    results_table = pd.DataFrame({
        "coef":    model.params,
        "HC3_se":  model.bse,
        "t":       model.tvalues,
        "p-value": model.pvalues,
        "CI_low":  model.conf_int()[0],
        "CI_high": model.conf_int()[1],
    }).round(4)

    print(f"\n── Cross-section OLS: {year} ── N={len(d)} countries ──")
    print(f"R²: {model.rsquared:.3f}")
    print(results_table)

    return model

In [18]:
model_2005_ind = cross_section_ols(df_clean_ind, 2005)


── Cross-section OLS: 2005 ── N=175 countries ──
R²: 0.742
                       coef  HC3_se        t  p-value   CI_low  CI_high
Intercept          -12.2290  0.9884 -12.3724   0.0000 -14.1662 -10.2917
log_gdp              1.1927  0.0510  23.3663   0.0000   1.0927   1.2928
log_industry_share   0.6511  0.2765   2.3546   0.0185   0.1091   1.1931


## 4. First difference model, with time trend, 2 year lags

In [19]:
from linearmodels.panel import PanelOLS

def first_diff_model(df, lags=0):
    """
    First difference model using PanelOLS.
    TimeEffects replaces C(year) dummies.
    """
    # ── prep ──────────────────────────────────────────────────────────────────
    d = (df.query("co2_pc > 0 and gdp_pc > 0 and industry_share > 0")
           .assign(log_co2=lambda x: np.log(x["co2_pc"]),
                   log_gdp=lambda x: np.log(x["gdp_pc"]),
                   log_industry_share=lambda x: np.log(x['industry_share']))
           .sort_values(["iso3", "year"]))

    # ── first differences within each country ─────────────────────────────────
    d["dlog_co2"] = d.groupby("iso3")["log_co2"].diff()
    d["dlog_gdp"] = d.groupby("iso3")["log_gdp"].diff()
    d["dlog_industry_share"] = d.groupby("iso3")["log_industry_share"].diff()

    # ── lags ──────────────────────────────────────────────────────────────────
    formula_vars = "dlog_gdp + dlog_industry_share"
    if lags > 0:
        for lag in range(1, lags + 1):
            d[f"dlog_gdp_lag{lag}"] = d.groupby("iso3")["dlog_gdp"].shift(lag)
            d[f"dlog_industry_share_lag{lag}"] = d.groupby("iso3")["dlog_industry_share"].shift(lag)
            formula_vars += f" + dlog_gdp_lag{lag} + dlog_industry_share_lag{lag}"

    # ── dropna + set MultiIndex (required by linearmodels) ────────────────────
    drop_cols = ["dlog_co2", "dlog_gdp", "dlog_industry_share"] + [f"dlog_gdp_lag{lag}" for lag in range(1, lags+1)] + [f"dlog_industry_share_lag{lag}" for lag in range(1, lags+1)]
    d = (d.dropna(subset=drop_cols)
          .set_index(["iso3", "year"]))  # PanelOLS requires MultiIndex

    model = PanelOLS.from_formula(
        f"dlog_co2 ~ {formula_vars} + TimeEffects",
        data=d
    ).fit(cov_type="clustered", cluster_entity=True)

    print(f"\n── FD Model (PanelOLS) | lags={lags} ──")
    print(f"N obs: {model.nobs:,}  |  N countries: {d.index.get_level_values('iso3').nunique()}")
    print(f"R² (within): {model.rsquared:.3f}")
    print(model.summary.tables[1])

    return model

In [20]:
model_fd2_ind = first_diff_model(df_clean_ind, lags=2)


── FD Model (PanelOLS) | lags=2 ──
N obs: 5,123  |  N countries: 184
R² (within): 0.034
                                    Parameter Estimates                                     
                          Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
--------------------------------------------------------------------------------------------
dlog_gdp                     0.2679     0.0473     5.6660     0.0000      0.1752      0.3606
dlog_industry_share          0.0251     0.0244     1.0295     0.3033     -0.0227      0.0729
dlog_gdp_lag1                0.0558     0.0432     1.2925     0.1963     -0.0289      0.1405
dlog_industry_share_lag1     0.0197     0.0156     1.2617     0.2071     -0.0109      0.0503
dlog_gdp_lag2                0.0513     0.0232     2.2152     0.0268      0.0059      0.0967
dlog_industry_share_lag2     0.0347     0.0137     2.5391     0.0111      0.0079      0.0615


## 6. Fixed effect model with time and country fixed effect

In [21]:
# from linearmodels.panel import PanelOLS

def fixed_effects_model(df):
    """
    Fixed effects model with time and country fixed effects.
    Equivalent to: log_co2 ~ log_gdp + EntityEffects + TimeEffects
    """
    # ── prep ──────────────────────────────────────────────────────────────────
    d = (df.query("co2_pc > 0 and gdp_pc > 0 and industry_share > 0")
           .assign(log_co2=lambda x: np.log(x["co2_pc"]),
                   log_gdp=lambda x: np.log(x["gdp_pc"]),
                   log_industry_share=lambda x: np.log(x["industry_share"])
                   )
           .dropna(subset=["log_co2", "log_gdp", "log_industry_share"])
           .set_index(["iso3", "year"]))  # linearmodels requires MultiIndex

    model = PanelOLS.from_formula(
        "log_co2 ~ log_gdp + log_industry_share + EntityEffects + TimeEffects",
        data=d
    ).fit(cov_type="clustered", cluster_entity=True)

    print("\n── Fixed Effects Model | Country + Time FE ──")
    print(f"N obs: {model.nobs:,}  |  N countries: {d.index.get_level_values('iso3').nunique()}")
    print(f"R² (within): {model.rsquared:.3f}")
    print(model.summary.tables[1])

    return model

In [22]:
model_fe_ind = fixed_effects_model(df_clean_ind)


── Fixed Effects Model | Country + Time FE ──
N obs: 5,675  |  N countries: 184
R² (within): 0.126
                                 Parameter Estimates                                  
                    Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
--------------------------------------------------------------------------------------
log_gdp                0.3898     0.0933     4.1790     0.0000      0.2069      0.5726
log_industry_share     0.3238     0.0850     3.8088     0.0001      0.1571      0.4904
